# E2 -- 40-mode Gaussian mixture (2D): run notebook

**This notebook runs and saves. It does not typeset figures.**

Every official metric is computed here, at run time, and written into each run's `metrics_timeseries.csv` and `cost_timeseries.csv`. The companion notebook `E2_mog40_plot.ipynb` reads those numbers and never recomputes them.

**Run All executes the single default full configuration.** There is exactly one configuration for this experiment, `configs/experiments/E2.yaml` -- there is no smoke, dev, reduced, or production profile to choose between. Lowering the particle count for local debugging is an explicit temporary edit, never a second committed profile.

Each variant is saved the moment it finishes, into its own atomically renamed run directory, so a variant that fails leaves the earlier ones untouched.

In [1]:
import sys

sys.path.insert(0, "..")  # importable when launched from notebooks/

from src.pipeline import load_experiment, run_variants_and_save

## Target, reference, and cost calibration

The reference is built **once** and reused by every method. It does not depend on any method parameter, so it is **never rebuilt per method, per hyperparameter value, or per canonical/tamed variant**; a cached reference on disk is loaded instead of being recomputed.

The force-equivalent-evaluation (FEE) calibration is measured once per device in the same way, and every run in this experiment is costed against that one calibration. The device is resolved automatically -- no device index is pinned in this notebook.

In [2]:
experiment = load_experiment("E2", device="auto")

reference = experiment.ensure_reference()
fee = experiment.ensure_fee_calibration()

described = reference.describe()
print(f"reference: kind={described.get('kind', described.get('method'))}  hash={experiment.reference_hash}")
print(f"FEE:       unit={fee.cost_unit}  hash={fee.hash}")

reference: kind=exact_gaussian_mixture  hash=1ed7f635d25fe5cc00b6acb78b3f3fe4
FEE:       unit=amortized_time_per_configuration  hash=076536868d101ed21862549ee5a0e135


## ULA

Every taming-capable method runs **both** a canonical and a tamed variant. `run_variants_and_save` expands each entry of `variants` into those two runs by itself, so a notebook never passes `tame`. The two variants are **calibrated separately** -- each one gets its own step size from its own `dt` refinement -- and each is saved as its own run directory.

In [3]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="ULA",
    # `run_variants_and_save` expands each entry below into a
    # canonical and a tamed run, so `tame` is never passed here.
    variants=[{}],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

[ULA, canonical] saved to /home/zheyuanlai/levy-sampling/results/E2_mog40/runs/ULA/ULA-canonical-dt0.01-20260806T213400388056Z


[ULA, tamed] saved to /home/zheyuanlai/levy-sampling/results/E2_mog40/runs/ULA/ULA-tamed-dt0.01-20260806T213406601880Z


[{'variant_label': 'ULA, canonical',
  'status': 'complete',
  'run_id': 'ULA-canonical-dt0.01-20260806T213400388056Z',
  'run_directory': '/home/zheyuanlai/levy-sampling/results/E2_mog40/runs/ULA/ULA-canonical-dt0.01-20260806T213400388056Z',
  'dt': 0.01,
  'calibration_hash': 'c2a90bd32c53acdb3a12f96e2e3ee603',
  'fee_calibration_hash': '076536868d101ed21862549ee5a0e135',
  'n_metric_rows': 884,
  'n_snapshots': 4},
 {'variant_label': 'ULA, tamed',
  'status': 'complete',
  'run_id': 'ULA-tamed-dt0.01-20260806T213406601880Z',
  'run_directory': '/home/zheyuanlai/levy-sampling/results/E2_mog40/runs/ULA/ULA-tamed-dt0.01-20260806T213406601880Z',
  'dt': 0.01,
  'calibration_hash': '00b5940a5bebeff67d3aa5f802f937a1',
  'fee_calibration_hash': '076536868d101ed21862549ee5a0e135',
  'n_metric_rows': 884,
  'n_snapshots': 4}]

## MALA

MALA supports taming, so it also runs both variants. Tamed MALA implements the actual tamed proposal density in the Metropolis-Hastings ratio; it is a genuine second sampler, not a relabelled copy of the canonical run.

In [4]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="MALA",
    variants=[{}],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

[MALA, canonical] NOT CALIBRATABLE: unstable at every timestep tried (mh_acceptance_outside_target_band, temporal_ess_fraction) though it improves as the timestep shrinks


[MALA, tamed] NOT CALIBRATABLE: unstable at every timestep tried (mh_acceptance_outside_target_band, temporal_ess_fraction) though it improves as the timestep shrinks


[{'variant_label': 'MALA, canonical',
  'method': 'MALA',
  'status': 'uncalibratable',
  'diagnosis': 'unstable at every timestep tried (mh_acceptance_outside_target_band, temporal_ess_fraction) though it improves as the timestep shrinks',
  'calibration_kind': 'timestep',
  'calibration_table': [{'dt': 10.24,
    'pass': False,
    'stability_problems': [('temporal_ess_fraction', nan)],
    'agreement_failures': [],
    'summary': {'n_steps': 1,
     'nonfinite_fraction': 0.0,
     'boundary_reject_fraction': 0.0,
     'temporal_ess': nan,
     'temporal_ess_fraction': nan,
     'temporal_ess_draws_per_seed': 1,
     'n_effective': 1024,
     'summary_mean': 10.89934195996372,
     'summary_mean_se': 0.023374159129328366,
     'summary_abs_mean': 10.89934195996372,
     'summary_abs_mean_se': 0.023374159129328366,
     'summary_median': 10.899733095795558,
     'summary_median_se': 0.023878538195882985,
     'summary_iqr': 0.8613031773341824,
     'summary_iqr_se': 0.0285939322648512

## FLA

The three stability indices are this experiment's default grid in `configs/registry.yaml`. All three run from this one cell and save as separate variants, and each of them is expanded into a canonical and a tamed run.

In [5]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="FLA",
    variants=[
        {"alpha": 1.6}, {"alpha": 1.7}, {"alpha": 1.8},
    ],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

[FLA alpha=1.6, canonical] saved to /home/zheyuanlai/levy-sampling/results/E2_mog40/runs/FLA/FLA-alpha1.6-canonical-dt0.01-20260806T213416220673Z


[FLA alpha=1.6, tamed] saved to /home/zheyuanlai/levy-sampling/results/E2_mog40/runs/FLA/FLA-alpha1.6-tamed-dt0.01-20260806T213423718008Z


[FLA alpha=1.7, canonical] saved to /home/zheyuanlai/levy-sampling/results/E2_mog40/runs/FLA/FLA-alpha1.7-canonical-dt0.01-20260806T213430806534Z


[FLA alpha=1.7, tamed] saved to /home/zheyuanlai/levy-sampling/results/E2_mog40/runs/FLA/FLA-alpha1.7-tamed-dt0.01-20260806T213438061412Z


[FLA alpha=1.8, canonical] saved to /home/zheyuanlai/levy-sampling/results/E2_mog40/runs/FLA/FLA-alpha1.8-canonical-dt0.01-20260806T213445186034Z


[FLA alpha=1.8, tamed] saved to /home/zheyuanlai/levy-sampling/results/E2_mog40/runs/FLA/FLA-alpha1.8-tamed-dt0.01-20260806T213452545680Z


[{'variant_label': 'FLA alpha=1.6, canonical',
  'status': 'complete',
  'run_id': 'FLA-alpha1.6-canonical-dt0.01-20260806T213416220673Z',
  'run_directory': '/home/zheyuanlai/levy-sampling/results/E2_mog40/runs/FLA/FLA-alpha1.6-canonical-dt0.01-20260806T213416220673Z',
  'dt': 0.01,
  'calibration_hash': 'be3615f1f4e05ec63a0824000882f1fd',
  'fee_calibration_hash': '076536868d101ed21862549ee5a0e135',
  'n_metric_rows': 884,
  'n_snapshots': 4},
 {'variant_label': 'FLA alpha=1.6, tamed',
  'status': 'complete',
  'run_id': 'FLA-alpha1.6-tamed-dt0.01-20260806T213423718008Z',
  'run_directory': '/home/zheyuanlai/levy-sampling/results/E2_mog40/runs/FLA/FLA-alpha1.6-tamed-dt0.01-20260806T213423718008Z',
  'dt': 0.01,
  'calibration_hash': '14117dc81d2df61b8d90b1ba264c24b4',
  'fee_calibration_hash': '076536868d101ed21862549ee5a0e135',
  'n_metric_rows': 884,
  'n_snapshots': 4},
 {'variant_label': 'FLA alpha=1.7, canonical',
  'status': 'complete',
  'run_id': 'FLA-alpha1.7-canonical-dt0.0

## ULD

ULD is the method; BAOAB is the integrator it is discretised with. Runs, manifests, and legends say ULD.

In [6]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="ULD",
    variants=[{}],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

[ULD gamma=1, canonical] saved to /home/zheyuanlai/levy-sampling/results/E2_mog40/runs/ULD/ULD-gamma1-canonical-dt0.01-20260806T213458638385Z


[ULD gamma=1, tamed] saved to /home/zheyuanlai/levy-sampling/results/E2_mog40/runs/ULD/ULD-gamma1-tamed-dt0.01-20260806T213504964791Z


[{'variant_label': 'ULD gamma=1, canonical',
  'status': 'complete',
  'run_id': 'ULD-gamma1-canonical-dt0.01-20260806T213458638385Z',
  'run_directory': '/home/zheyuanlai/levy-sampling/results/E2_mog40/runs/ULD/ULD-gamma1-canonical-dt0.01-20260806T213458638385Z',
  'dt': 0.01,
  'calibration_hash': '0d3c61f3875bf320ec075ac3f4bbedab',
  'fee_calibration_hash': '076536868d101ed21862549ee5a0e135',
  'n_metric_rows': 884,
  'n_snapshots': 4},
 {'variant_label': 'ULD gamma=1, tamed',
  'status': 'complete',
  'run_id': 'ULD-gamma1-tamed-dt0.01-20260806T213504964791Z',
  'run_directory': '/home/zheyuanlai/levy-sampling/results/E2_mog40/runs/ULD/ULD-gamma1-tamed-dt0.01-20260806T213504964791Z',
  'dt': 0.01,
  'calibration_hash': 'aba7a8ac5a8a1e07203c69b549d3fe1a',
  'fee_calibration_hash': '076536868d101ed21862549ee5a0e135',
  'n_metric_rows': 884,
  'n_snapshots': 4}]

## PT

Parallel tempering. The replica ladder is tuned by the calibration step that `run_variants_and_save` invokes, not here, and the tuned ladder is written into the run's `calibration.json`.

In [7]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="PT",
    variants=[{}],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

[PT n_swap=10, canonical] NOT CALIBRATABLE: unstable at every timestep tried (boundary_reject_fraction, mh_acceptance_outside_target_band) and it does not improve as the timestep shrinks


[PT n_swap=10, tamed] saved to /home/zheyuanlai/levy-sampling/results/E2_mog40/runs/PT/PT-n_swap10-tamed-dt5.12-20260806T213632834708Z


[{'variant_label': 'PT n_swap=10, canonical',
  'method': 'PT',
  'status': 'uncalibratable',
  'diagnosis': 'unstable at every timestep tried (boundary_reject_fraction, mh_acceptance_outside_target_band) and it does not improve as the timestep shrinks',
  'calibration_kind': 'timestep',
  'calibration_table': [{'dt': 10.24,
    'pass': False,
    'stability_problems': [('boundary_reject_fraction', 0.02099609375)],
    'agreement_failures': [{'key': 'summary_iqr',
      'coarse': 0.94552851,
      'fine': 1.1737983,
      'difference': 0.22826978,
      'allowance': 0.2282434}],
    'summary': {'n_steps': 1,
     'nonfinite_fraction': 0.0,
     'boundary_reject_fraction': 0.02099609375,
     'n_effective': 1024,
     'summary_mean': 10.941646305521225,
     'summary_mean_se': 0.02393566938675817,
     'summary_abs_mean': 10.941646305521225,
     'summary_abs_mean_se': 0.02393566938675817,
     'summary_median': 10.984029090408129,
     'summary_median_se': 0.024943707873549045,
     's

## Raw-CP

The same compound-Poisson jump process with the Levy score correction switched off. It does not preserve the target, so it is the control arm that isolates what the score correction buys, not a competitive baseline.

In [8]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="Raw-CP",
    variants=[{}],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

[Raw-CP, canonical] saved to /home/zheyuanlai/levy-sampling/results/E2_mog40/runs/Raw-CP/Raw-CP-canonical-dt0.01-20260806T213639979566Z


[Raw-CP, tamed] saved to /home/zheyuanlai/levy-sampling/results/E2_mog40/runs/Raw-CP/Raw-CP-tamed-dt0.01-20260806T213647525994Z


[{'variant_label': 'Raw-CP, canonical',
  'status': 'complete',
  'run_id': 'Raw-CP-canonical-dt0.01-20260806T213639979566Z',
  'run_directory': '/home/zheyuanlai/levy-sampling/results/E2_mog40/runs/Raw-CP/Raw-CP-canonical-dt0.01-20260806T213639979566Z',
  'dt': 0.01,
  'calibration_hash': 'fdc1ac20a9de2a1bf0b9c5368256d009',
  'fee_calibration_hash': '076536868d101ed21862549ee5a0e135',
  'n_metric_rows': 884,
  'n_snapshots': 4},
 {'variant_label': 'Raw-CP, tamed',
  'status': 'complete',
  'run_id': 'Raw-CP-tamed-dt0.01-20260806T213647525994Z',
  'run_directory': '/home/zheyuanlai/levy-sampling/results/E2_mog40/runs/Raw-CP/Raw-CP-tamed-dt0.01-20260806T213647525994Z',
  'dt': 0.01,
  'calibration_hash': 'e1c3f98c3ac166ae0e9deb02271d6ea0',
  'fee_calibration_hash': '076536868d101ed21862549ee5a0e135',
  'n_metric_rows': 884,
  'n_snapshots': 4}]

## LSC-CP

Compound-Poisson jumps with the full deterministic-quadrature Levy score correction.

In [9]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="LSC-CP",
    variants=[{}],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

[LSC-CP, canonical] NOT CALIBRATABLE: unstable at every timestep tried (boundary_reject_fraction) and it does not improve as the timestep shrinks


[LSC-CP, tamed] saved to /home/zheyuanlai/levy-sampling/results/E2_mog40/runs/LSC-CP/LSC-CP-tamed-dt0.01-20260806T214630457669Z


[{'variant_label': 'LSC-CP, canonical',
  'method': 'LSC-CP',
  'status': 'uncalibratable',
  'diagnosis': 'unstable at every timestep tried (boundary_reject_fraction) and it does not improve as the timestep shrinks',
  'calibration_kind': 'timestep',
  'calibration_table': [{'dt': 0.01,
    'pass': False,
    'stability_problems': [('boundary_reject_fraction', 0.7399765625)],
    'agreement_failures': [],
    'summary': {'n_steps': 625,
     'nonfinite_fraction': 0.0,
     'boundary_reject_fraction': 0.7399765625,
     'n_effective': 1024,
     'summary_mean': 11.46816774378738,
     'summary_mean_se': 0.4428783084737146,
     'summary_abs_mean': 13.598199607983162,
     'summary_abs_mean_se': 0.3616761485909181,
     'summary_median': 10.199048801131136,
     'summary_median_se': 0.6053518442546504,
     'summary_iqr': 17.70041993089641,
     'summary_iqr_se': 0.4289936805438713,
     'energy_mean': 4.999977167334011,
     'energy_mean_se': 0.2412369644314437,
     'energy_abs_mean':

## LSC-CP-RA

`A` is the **iid Monte Carlo bank size of one estimator family**, LSC-CP-RA. `A = 1, 4, 8` are variants of that single family, not three separate methods, and **all of them run from this one cell** and save as separate variants.

The bank holds `A` displacements drawn iid from the full normalised jump law `rho = nu / lambda`, and **the same bank drives both the score and the compound-Poisson increment**.

In [10]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="LSC-CP-RA",
    variants=[
        {"A": 1}, {"A": 4}, {"A": 8},
    ],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

[LSC-CP-RA, canonical] NOT CALIBRATABLE: unstable at every timestep tried (boundary_reject_fraction) and it does not improve as the timestep shrinks


[LSC-CP-RA, tamed] saved to /home/zheyuanlai/levy-sampling/results/E2_mog40/runs/LSC-CP-RA/LSC-CP-RA-A1-tamed-dt0.01-20260806T214659071304Z


[LSC-CP-RA (A=4), canonical] NOT CALIBRATABLE: unstable at every timestep tried (boundary_reject_fraction) and it does not improve as the timestep shrinks


[LSC-CP-RA (A=4), tamed] saved to /home/zheyuanlai/levy-sampling/results/E2_mog40/runs/LSC-CP-RA/LSC-CP-RA-A4-tamed-dt0.01-20260806T214732326680Z


[LSC-CP-RA (A=8), canonical] NOT CALIBRATABLE: unstable at every timestep tried (boundary_reject_fraction) and it does not improve as the timestep shrinks


[LSC-CP-RA (A=8), tamed] saved to /home/zheyuanlai/levy-sampling/results/E2_mog40/runs/LSC-CP-RA/LSC-CP-RA-A8-tamed-dt0.01-20260806T214813993755Z


[{'variant_label': 'LSC-CP-RA, canonical',
  'method': 'LSC-CP-RA',
  'status': 'uncalibratable',
  'diagnosis': 'unstable at every timestep tried (boundary_reject_fraction) and it does not improve as the timestep shrinks',
  'calibration_kind': 'timestep',
  'calibration_table': [{'dt': 0.01,
    'pass': False,
    'stability_problems': [('boundary_reject_fraction', 0.386715625)],
    'agreement_failures': [],
    'summary': {'n_steps': 625,
     'nonfinite_fraction': 0.0,
     'boundary_reject_fraction': 0.386715625,
     'n_effective': 1024,
     'summary_mean': 3.081174182657035,
     'summary_mean_se': 1.153908122602663,
     'summary_abs_mean': 32.40890358832242,
     'summary_abs_mean_se': 0.5383935580483938,
     'summary_median': 3.932409312903668,
     'summary_median_se': 1.7392569337154593,
     'summary_iqr': 67.30996217709941,
     'summary_iqr_se': 2.3770484205004108,
     'energy_mean': 18.463627825457444,
     'energy_mean_se': 0.4782742508104306,
     'energy_abs_mean

## Rebuild the catalog

`catalog.csv` is a **derived index** over the run manifests. It is never written by a worker mid-run, so concurrent runs never contend for it, and it can be rebuilt at any time by rescanning the manifests -- a lost or stale catalog costs nothing. Only runs that verify (manifest present, `COMPLETE` present, hashes matching) are admitted.

In [11]:
from src.catalog import write_catalog

report = write_catalog(experiment.paths.experiment_dir)
print(f"catalog rebuilt: {report['n_runs']} runs, {report['n_rejected']} rejected")

catalog rebuilt: 72 runs, 32 rejected
